# 02 · Bronze: load cost exports with COPY INTO
Loads the data files from one Azure cost export (already landed in a volume by `01_sync_from_storage`)
into a bronze Delta table, then checks every loaded run against its `manifest.json`.

**Before running:** `00_setup` and `01_sync_from_storage` have run, and the files are in the volume.
**Safe to re-run:** COPY INTO skips files it has already loaded.

In [0]:
# Config + Parameters
dbutils.widgets.text("catalog", "finops")
dbutils.widgets.text("export_name", "george-test-focus-cost")
dbutils.widgets.text("target_table", "focus_costs")

CATALOG     = dbutils.widgets.get("catalog")
EXPORT_NAME = dbutils.widgets.get("export_name")

SOURCE_DIR   = f"/Volumes/{CATALOG}/raw/real_exports/exports/{EXPORT_NAME}/"
TARGET_TABLE = f"{CATALOG}.bronze.{dbutils.widgets.get('target_table')}"

In [0]:
# Walk the export folder and list every file with its size
def list_files(path):
    for f in dbutils.fs.ls(path):
        if f.isDir():
            yield from list_files(f.path)
        else:
            yield (f.path, f.size)

display(spark.createDataFrame(list(list_files(SOURCE_DIR)), "path string, size long"))

In [0]:
# Both are no-ops if they already exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze COMMENT 'Raw rows loaded from landed export files'")

# No columns yet; COPY INTO with mergeSchema creates them from the files
spark.sql(f"CREATE TABLE IF NOT EXISTS {TARGET_TABLE} COMMENT 'Raw cost export rows, loaded with COPY INTO'")

In [0]:
# Load data into bronze tables using COPY INTO
result = spark.sql(f"""
  COPY INTO {TARGET_TABLE}
  FROM (
    SELECT *,
           _metadata.file_path AS source_file,   -- which file each row came from
           current_timestamp() AS ingested_at    -- when we loaded it
    FROM '{SOURCE_DIR}'
  )
  FILEFORMAT = CSV
  PATTERN = '*/*/part_*.csv.gz'                  -- period folder / run folder / data file; skips manifests
  FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')   -- bronze keeps everything as strings
  COPY_OPTIONS ('mergeSchema' = 'true')
""")
display(result)   # num_inserted_rows = 0 on a re-run means nothing new to load

In [0]:
# Check every run against its own manifest 

import json
from pyspark.sql import functions as F

# Count rows per run folder (the folder holding each data file and its manifest)
loaded = (spark.table(TARGET_TABLE)
          .withColumn("run_folder", F.regexp_extract("source_file", r"^(.*)/[^/]+$", 1))
          .groupBy("run_folder").count()
          .collect())

if not loaded:
    raise ValueError(f"{TARGET_TABLE} is empty")

problems = []
for row in loaded:
    expected = json.loads(dbutils.fs.head(f"{row.run_folder}/manifest.json"))["dataRowCount"]
    print(f"run {row.run_folder.rsplit('/', 1)[-1]}: manifest {expected}, table {row['count']}")
    if expected != row["count"]:
        problems.append(row.run_folder)

# Raise error if failed so a job run shows as failed
if problems:
    raise ValueError(f"row counts don't match the manifest for: {problems}")